# Gift-Eval Ablation Analysis

This notebook analyzes the effects of a TSFM's performance across different datasets.

**Ablation Levels:**
- Level 0: Original model (no ablation)
- Level 1: Ablate layers 10, heads-all
- Level 2: Ablate layers 10-11, heads-all
- Level 3: Ablate layers 10-11-12, heads-all
- Level 4: Ablate layers 10-11-12-13, heads-all
- Level 5: Ablate layers 10-11-12-13-14, heads-all
- Level 6: Ablate layers 10-11-12-13-14-15, heads-all


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
HOME_DIR = os.getenv("HOME")

In [ ]:
root_dir = os.path.join(HOME_DIR, "tsfm-lens")

In [ ]:
model_name = "Toto"

model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
}

In [ ]:
# Select the metric to analyze
SELECTED_METRIC = "eval_metrics/MASE[0.5]"
# SELECTED_METRIC = "eval_metrics/sMAPE[0.5]"

metric_pretty_name = "".join(c for c in SELECTED_METRIC.split("/")[-1] if c.isalpha())

print(f"Analyzing metric: {SELECTED_METRIC}")
print(f"Metric name: {metric_pretty_name}")


In [ ]:
save_figs = True
figs_save_dir = os.path.join(root_dir, "figs", f"gift-eval_{metric_pretty_name}", model_to_dir_mapping[model_name])
if not os.path.exists(figs_save_dir):
    os.makedirs(figs_save_dir)
if save_figs:
    print(f"Saving figures to: {figs_save_dir}")

## Load Data


In [ ]:
# Heads and MLP ablation
if model_name == "TimesFM 2.5":
    selected_layers_lst = [
        11,
        [10, 11],
        [10, 11, 12],
        [10, 11, 12, 13],
        [9, 10, 11, 12, 13],
        [7, 8, 9, 10, 11, 12],
        [7, 8, 9, 10, 11, 12, 13],
    ]
    n_divisor = 20  # number of layers

    head_mlp_ablation_files = {}
    head_mlp_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        head_mlp_ablation_files[num_layers] = f"head-mlp_layers_{selected_layers_str}_heads-all_all_results.csv"
        head_mlp_ablation_files = {
            round(k, 3) if isinstance(k, float) else k: v for k, v in head_mlp_ablation_files.items()
        }
elif model_name == "Chronos Bolt":
    selected_layers_lst = [2, [1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]
    n_divisor = 12  # number of layers

    head_mlp_ablation_files = {}
    head_mlp_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        head_mlp_ablation_files[num_layers] = f"head-mlp_layers_{selected_layers_str}_heads-all_all_results.csv"
        head_mlp_ablation_files = {
            round(k, 3) if isinstance(k, float) else k: v for k, v in head_mlp_ablation_files.items()
        }
elif model_name == "Chronos":
    # Chronos
    selected_layers_lst = []
    n_divisor = 12  # number of layers

    head_mlp_ablation_files = {}
    head_mlp_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        head_mlp_ablation_files[num_layers] = f"head-mlp_layers_{selected_layers_str}_heads-all_all_results.csv"
        head_mlp_ablation_files = {
            round(k, 3) if isinstance(k, float) else k: v for k, v in head_mlp_ablation_files.items()
        }
elif model_name == "Toto":
    # selected_layers_lst = [9, [9, 10], [9, 10, 11], [8, 9, 10, 11], [7, 8, 9, 10, 11]]
    selected_layers_lst = [8, [9, 10], [3, 4, 5], [5, 6, 7, 8], [4, 5, 6, 7, 8]]
    n_divisor = 12  # number of layers

    head_mlp_ablation_files = {}
    head_mlp_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        head_mlp_ablation_files[num_layers] = f"head-mlp_layers_{selected_layers_str}_heads-all_all_results.csv"
        head_mlp_ablation_files = {
            round(k, 3) if isinstance(k, float) else k: v for k, v in head_mlp_ablation_files.items()
        }
else:
    raise ValueError(f"Model {model_name} not supported yet")


In [ ]:
head_mlp_ablation_files

In [ ]:
# Heads ablation
if model_name == "TimesFM 2.5":
    selected_layers_lst = [
        10,
        [10, 11],
        [10, 11, 12],
        [10, 11, 12, 13],
        [7, 8, 9, 10, 11],
        [7, 8, 9, 10, 11, 12],
    ]

    n_divisor = 20  # number of layers

    head_ablation_files = {}
    head_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        head_ablation_files[num_layers] = f"head_layers_{selected_layers_str}_heads-all_all_results.csv"
        head_ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in head_ablation_files.items()}
elif model_name == "Chronos Bolt":
    # Chronos Bolt
    selected_layers_lst = [2, [1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]
    n_divisor = 12  # number of layers

    head_ablation_files = {}
    head_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        head_ablation_files[num_layers] = f"head_layers_{selected_layers_str}_heads-all_all_results.csv"
        head_ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in head_ablation_files.items()}

elif model_name == "Chronos":
    # Chronos
    selected_layers_lst = [[7, 8], [5, 6, 7], [5, 6, 7, 8]]
    n_divisor = 12  # number of layers

    head_ablation_files = {}
    head_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        head_ablation_files[num_layers] = f"head_layers_{selected_layers_str}_heads-all_all_results.csv"
        head_ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in head_ablation_files.items()}
elif model_name == "Toto":
    # selected_layers_lst = [9, [9, 10], [9, 10, 11], [8, 9, 10, 11], [7, 8, 9, 10, 11]]
    selected_layers_lst = [9, [9, 10], [9, 10, 11], [2, 8, 9, 10], [7, 8, 9, 10, 11]]
    n_divisor = 12  # number of layers

    head_ablation_files = {}
    head_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        head_ablation_files[num_layers] = f"head_layers_{selected_layers_str}_heads-all_all_results.csv"
        head_ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in head_ablation_files.items()}

else:
    raise ValueError(f"Model {model_name} not supported yet")


In [ ]:
head_ablation_files

In [ ]:
if model_name == "TimesFM 2.5":
    # MLP ablation
    selected_layers_lst = [
        9,
        [8, 9],
        [7, 8, 9],
        [6, 7, 8, 9],
        [8, 9, 10, 11, 12],
        [8, 9, 10, 11, 12, 13],
        [7, 8, 9, 10, 11, 12, 13],
    ]
    n_divisor = 20  # number of layers

    mlp_ablation_files = {}
    mlp_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        mlp_ablation_files[num_layers] = f"mlp_layers_{selected_layers_str}_heads-all_all_results.csv"
        mlp_ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in mlp_ablation_files.items()}
elif model_name == "Chronos Bolt":
    selected_layers_lst = [8, [2, 3], [3, 4, 5], [3, 4, 5, 6], [2, 3, 4, 5, 6], [3, 4, 5, 6, 7, 8]]
    n_divisor = 12  # number of layers

    mlp_ablation_files = {}
    mlp_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        mlp_ablation_files[num_layers] = f"mlp_layers_{selected_layers_str}_heads-all_all_results.csv"
        mlp_ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in mlp_ablation_files.items()}

elif model_name == "Chronos":
    # Chronos
    selected_layers_lst = []
    n_divisor = 12  # number of layers

    mlp_ablation_files = {}
    mlp_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        mlp_ablation_files[num_layers] = f"mlp_layers_{selected_layers_str}_heads-all_all_results.csv"
        mlp_ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in mlp_ablation_files.items()}

elif model_name == "Toto":
    # selected_layers_lst = [9, [9, 10], [9, 10, 11], [8, 9, 10, 11], [7, 8, 9, 10, 11]]
    selected_layers_lst = [10, [8, 9], [6, 7, 8], [5, 6, 7, 8], [7, 8, 9, 10, 11]]
    n_divisor = 12  # number of layers

    mlp_ablation_files = {}
    mlp_ablation_files[0] = "original_all_results.csv"

    for selected_layers in selected_layers_lst:
        key = selected_layers
        num_layers = -1
        if isinstance(selected_layers, list):
            selected_layers_str = "-".join(map(str, selected_layers))
            key = f"Layers {selected_layers[0]}-{selected_layers[-1]}"
            num_layers = len(selected_layers)

        else:
            selected_layers_str = str(selected_layers)
            key = f"Layer {selected_layers_str}"
            num_layers = 1

        mlp_ablation_files[num_layers] = f"mlp_layers_{selected_layers_str}_heads-all_all_results.csv"
        mlp_ablation_files = {round(k, 3) if isinstance(k, float) else k: v for k, v in mlp_ablation_files.items()}

else:
    raise ValueError(f"Model {model_name} not supported yet")


In [ ]:
mlp_ablation_files

In [ ]:
rseeds_lst = [123, 99, 42, 688, 22, 33, 56, 78, 357, 234, 567, 890]
metrics_dir_dict = {
    rseed: os.path.join(root_dir, "results", model_to_dir_mapping[model_name], f"rseed-{rseed}") for rseed in rseeds_lst
}
print(f"Metrics directory: {metrics_dir_dict}")

In [ ]:
# Load and combine CSV files across ablation levels and rseeds
def load_ablation_data(ablated_files_dict, metrics_dir_dict):
    # Load and combine CSV files across ablation levels and rseeds
    # Strategy: For each level, compute geometric mean across datasets for each rseed,
    # then select the rseed with the lowest geometric mean
    best_filepaths_by_key = {}
    data_by_key = {}

    # Handle case where ablated_files_dict is a simple dict mapping level -> filepath
    # vs a nested dict mapping key -> {level -> filepath}
    if all(isinstance(v, str) for v in ablated_files_dict.values()):
        # Simple case: ablated_files_dict is {level: filepath}
        ablated_files_dict = {"default": ablated_files_dict}

    for key, ablation_files in ablated_files_dict.items():
        if key not in data_by_key:
            data_by_key[key] = {}
        if key not in best_filepaths_by_key:
            best_filepaths_by_key[key] = {}
        curr_data_frame = {}

        for level, filepath in ablation_files.items():
            # Dictionary to store data by rseed: {rseed: df}
            rseed_data = {}

            for rseed, metrics_dir in metrics_dir_dict.items():
                df_filepath = os.path.join(metrics_dir, filepath)
                if not os.path.exists(df_filepath):
                    print(f"Level {level}, rseed {rseed}: Skipping (not found)")
                    continue
                print(f"Level {level}, rseed {rseed}: Loading")
                df = pd.read_csv(df_filepath)
                # check if df has 97 rows; otherwise it is incomplete and should be skipped
                if len(df) != 97:
                    print(f"Level {level}, rseed {rseed}: Skipping (incomplete)")
                    continue
                df["ablation_level"] = level
                rseed_data[rseed] = (df, df_filepath)

            if not rseed_data:
                print(f"Level {level}: No data found, skipping")
                continue

            # For each rseed, compute geometric mean of metrics across all datasets
            # Then select the rseed with the lowest geometric mean
            metric_cols = None
            rseed_gmeans = {}

            for rseed, (df, df_filepath) in rseed_data.items():
                if metric_cols is None:
                    # Select only numeric columns that are metrics (exclude metadata columns)
                    exclude_cols = [
                        "dataset",
                        "model",
                        "domain",
                        "num_variates",
                        "num_datasets",
                        "ablation_level",
                        "frequency",
                    ]
                    metric_cols = [
                        col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])
                    ]

                # Compute geometric mean for the selected metric only
                selected_metric_gmean = gmean(df[SELECTED_METRIC])

                # Store the geometric mean of the selected metric as the overall score
                rseed_gmeans[rseed] = {
                    f"gmean_{SELECTED_METRIC}": selected_metric_gmean,
                    "filepath": df_filepath,
                    "df": df,
                }

            # Find rseed with lowest overall geometric mean
            best_rseed = min(rseed_gmeans.keys(), key=lambda r: rseed_gmeans[r][f"gmean_{SELECTED_METRIC}"])
            best_df = rseed_gmeans[best_rseed]["df"]
            best_filepath = rseed_gmeans[best_rseed]["filepath"]

            # Store the best filepath information
            best_filepaths_by_key[key][level] = {
                "best_rseed": best_rseed,
                "best_filepath": best_filepath,
                "available_rseeds": list(rseed_data.keys()),
                f"gmean_{SELECTED_METRIC}": rseed_gmeans[best_rseed][f"gmean_{SELECTED_METRIC}"],
                "num_rseeds_evaluated": len(rseed_data),
            }

            curr_data_frame[level] = best_df
            print(
                f"Level {level}: Best rseed={best_rseed} (gmean={rseed_gmeans[best_rseed][f'gmean_{SELECTED_METRIC}']:.6f}) from {len(rseed_data)} rseeds"
            )

        all_data_for_key = pd.concat(curr_data_frame.values(), ignore_index=True)
        data_by_key[key] = all_data_for_key
        print(f"\nTotal: {len(all_data_for_key)} rows, {all_data_for_key['dataset'].nunique()} unique datasets")

    # If we only had one key (default), return the data directly instead of nested
    if len(data_by_key) == 1 and "default" in data_by_key:
        return best_filepaths_by_key["default"], data_by_key["default"]

    return best_filepaths_by_key, data_by_key


# Load data for each ablation type
head_mlp_data_frames, head_mlp_all_data = load_ablation_data(head_mlp_ablation_files, metrics_dir_dict)
head_data_frames, head_all_data = load_ablation_data(head_ablation_files, metrics_dir_dict)
mlp_data_frames, mlp_all_data = load_ablation_data(mlp_ablation_files, metrics_dir_dict)

## Optional: Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


In [ ]:
if NORMALIZE_BY_SEASONAL_NAIVE:
    print("\nLoading seasonal naive baseline...")
    seasonal_naive_path = os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv")
    seasonal_naive_df = pd.read_csv(seasonal_naive_path)
    print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets")

    print("\nNormalizing data by seasonal naive baseline...")
    head_mlp_all_data = normalize_by_seasonal_naive(head_mlp_all_data, seasonal_naive_df)
    head_all_data = normalize_by_seasonal_naive(head_all_data, seasonal_naive_df)
    mlp_all_data = normalize_by_seasonal_naive(mlp_all_data, seasonal_naive_df)
else:
    print("\nSkipping normalization.")

## 1. Box Plot: Overall Performance Across Ablation Levels


In [ ]:
# Prepare data for box plot
head_mlp_box_data = head_mlp_all_data[["dataset", "ablation_level", SELECTED_METRIC]].copy()
mlp_box_data = mlp_all_data[["dataset", "ablation_level", SELECTED_METRIC]].copy()
head_all_box_data = head_all_data[["dataset", "ablation_level", SELECTED_METRIC]].copy()

# Remove any infinite or NaN values
head_mlp_box_data = head_mlp_box_data.replace([np.inf, -np.inf], np.nan).dropna()
mlp_box_data = mlp_box_data.replace([np.inf, -np.inf], np.nan).dropna()
head_all_box_data = head_all_box_data.replace([np.inf, -np.inf], np.nan).dropna()

## 2. Line Plot: Median Performance Trend


In [ ]:
# Calculate median and percentiles for each ablation level
def compute_stats(data, metric):
    return (
        data.groupby("ablation_level")[metric]
        .agg(
            [
                "median",
                "mean",
                lambda x: gmean(x),
                lambda x: np.exp(np.std(np.log(x)) / np.sqrt(len(x))),  # geometric standard error
                lambda x: x.quantile(0.25),
                lambda x: x.quantile(0.75),
            ]
        )
        .rename(columns={"<lambda_0>": "geom_mean", "<lambda_1>": "geom_ste", "<lambda_2>": "q25", "<lambda_3>": "q75"})
        .reset_index()
    )


head_mlp_stats_by_level = compute_stats(head_mlp_box_data, SELECTED_METRIC)
mlp_stats_by_level = compute_stats(mlp_box_data, SELECTED_METRIC)
head_all_stats_by_level = compute_stats(head_all_box_data, SELECTED_METRIC)


In [ ]:
# Plot each ablation type
# colors = plt.cm.get_cmap("Set1")(range(3))
# get default color cycle
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
print(colors)
plot_configs = [
    (head_all_stats_by_level, "Ablate Heads", colors[0], "^", 7, "-"),
    (mlp_stats_by_level, "Ablate MLPs", colors[1], "s", 7, "-"),
    (head_mlp_stats_by_level, "Ablate Heads + MLP", colors[2], "p", 9, "-"),
]

In [ ]:
# Create line plot with error bands
fig, ax = plt.subplots(figsize=(5, 4))

for stats, label, color, marker, markersize, linestyle in plot_configs:
    ax.plot(
        stats["ablation_level"],
        stats["geom_mean"],
        marker=marker,
        linewidth=2.5,
        markersize=markersize,
        markerfacecolor="None",
        markeredgecolor=color,
        markeredgewidth=2,
        label=label,
        color=color,
        linestyle=linestyle,
        alpha=1,
        zorder=3,
    )
    ax.fill_between(
        stats["ablation_level"],
        stats["geom_mean"] * (stats["geom_ste"] ** 0.33),
        stats["geom_mean"] / (stats["geom_ste"] ** 0.33),
        alpha=0.1,
        color=color,
        zorder=1,
    )

baseline_geom_mean = stats.loc[stats["ablation_level"] == 0, "geom_mean"].values[0]
# plot dashed line for baseline
ax.axhline(
    y=baseline_geom_mean,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Baseline ({baseline_geom_mean:.3f})",
    alpha=0.7,
    zorder=4,
)
# Add annotation at y-axis intersection
ax.text(
    0,
    baseline_geom_mean * 0.994,
    f"Baseline ({baseline_geom_mean:.3f})",
    color="red",
    fontsize=10,
    fontweight="bold",
    verticalalignment="top",
    horizontalalignment="left",
    zorder=100,
)
# plot dashed line for 1.002 * baseline_geom_mean
ax.axhline(
    y=1.02 * baseline_geom_mean,
    color="orangered",
    linestyle="--",
    linewidth=2,
)
# Add annotation at y-axis intersection
ax.text(
    0,
    (1.02 * baseline_geom_mean) * 1.004,
    f"Baseline + 2%",
    color="orangered",
    fontsize=10,
    fontweight="bold",
    verticalalignment="bottom",
    horizontalalignment="left",
    zorder=100,
)


# Formatting
ax.set_xlabel("Number of Layers Ablated (in contiguous blocks)" + " (%)", fontweight="bold")
ax.set_ylabel(f"{metric_pretty_name} (Geom Mean)", fontweight="bold")
# ax.set_title(f"ZA All Components in Contiguous Layers ({model_name})", fontweight="bold", pad=20)
ax.set_title(model_name, fontweight="bold", pad=10)
ax.grid(True, alpha=0.2)
ax.legend(loc="lower right", frameon=True, fontsize=9)

# Combine xticks from all datasets
all_levels = sorted(
    set(head_mlp_stats_by_level["ablation_level"].unique())
    | set(mlp_stats_by_level["ablation_level"].unique())
    | set(head_all_stats_by_level["ablation_level"].unique())
)
xticks = all_levels
ax.set_xticks(xticks)
ax.set_xticklabels([f"{int(x)}" if int(x) == x else f"{x:.1f}" for x in xticks], rotation=0)

ylow = 0.96 * baseline_geom_mean
yhigh = 1.12 * baseline_geom_mean
ax.set_ylim(ylow, yhigh)

plt.tight_layout()
save_path = os.path.join(figs_save_dir, f"comparison_line_plot_{model_name}.pdf")
if save_figs:
    plt.savefig(save_path, bbox_inches="tight")
    print(f"Saved comparison line plot to: {save_path}")
plt.show()

In [ ]:
rseeds_lst